In [1]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict
from qiskit_metal.toolbox_python.attr_dict import Dict
from qiskit_metal.qlibrary.core import QComponent

print(f"Qiskit-Metal version: {metal.__version__}")

Qiskit-Metal version: 0.5.3.post1


In [2]:
from qiskit_metal.qlibrary.user_components.circmong_singleJJ import CircmonG
print("CircmonG imported successfully.")
print()
print("Junction key in HFSS: 'rect_jj1'")
print("For component name 'TC1':")
print("  rect : JJ_rect_Lj_TC1_rect_jj1")
print("  line : JJ_Lj_TC1_rect_jj1_")

CircmonG imported successfully.

Junction key in HFSS: 'rect_jj1'
For component name 'TC1':
  rect : JJ_rect_Lj_TC1_rect_jj1
  line : JJ_Lj_TC1_rect_jj1_


In [3]:
# ── Create design ─────────────────────────────────────────────────────────
# Set chip size to match W (the metallic plane side).
# A small border is added so the HFSS airbox does not clip the component.
W_um = 800    # must match the W option below
H_um = 280 # substrate height

design = designs.DesignPlanar({}, True)
design.chips.main.size['size_x'] = f'{W_um}um'
design.chips.main.size['size_y'] = f'{W_um}um'
design.chips.main.size['size_z'] = f'{-H_um}um'
design.chips.main.size['center_x'] = '0um'
design.chips.main.size['center_y'] = '0um'

gui = MetalGUI(design)
print("Design created. MetalGUI opened.")

Design created. MetalGUI opened.


In [4]:
from qiskit_metal.qlibrary.user_components.circmong_singleJJ import CircmonG

circmon1 = CircmonG(design, 'TC1', options=dict(
    pos_x      = '0mm',
    pos_y      = '0mm',
    pad_r      = '200um',   # qubit island radius — primary Ec knob
    pad2pk_gap = '100um',   # slot width          — secondary Ec knob
    jj_width   = '20um',
    jj_angle   = 0,         # 0 deg = east (+x); single JJ only
    jj_in_pad  = '2um',
    jj_in_pk   = '2um',
))

In [5]:
design.overwrite_enabled = True
gui.rebuild()
gui.autoscale()

print("CircmonG built.")
print()
print(f"  pad_r      : {circmon1.options.pad_r}")
print(f"  pad2pk_gap : {circmon1.options.pad2pk_gap}")
print(f"  jj_width   : {circmon1.options.jj_width}")
print(f"  jj_angle   : {circmon1.options.jj_angle} deg")
print()
print("Junction table:")
print(design.qgeometry.tables['junction'][
    ['component','name','width','hfss_inductance','hfss_capacitance']])

CircmonG built.

  pad_r      : 200um
  pad2pk_gap : 100um
  jj_width   : 20um
  jj_angle   : 0 deg

Junction table:
  component      name  width hfss_inductance hfss_capacitance
0         1  rect_jj1   0.02              Lj               Cj


In [6]:
# This cell defines the hfss sim setup

from qiskit_metal.analyses.quantization import EPRanalysis

# ── Create EPRanalysis object ─────────────────────────────────────────────
eig_qb = EPRanalysis(design, "hfss")

# ── Simulation setup ──────────────────────────────────────────────────────
eig_qb.sim.setup.name          = 'Qbit_Setup'
eig_qb.sim.setup.n_modes       = 1
eig_qb.sim.setup.max_passes    = 15
eig_qb.sim.setup.max_delta_f   = 0.1    # convergence: 0.1% change in freq
eig_qb.sim.setup.min_freq_ghz  = 1.0
eig_qb.sim.setup.min_converged = 2

# ── Junction variable (single JJ) ─────────────────────────────────────────
# Single JJ: Ej = (Phi0/2pi)^2 / Lj  (no parallel combination factor)
# For f_01 ~ 3.5 GHz at Ec ~ 140 MHz: Ej ~ 11.8 GHz -> Lj ~ 13.5 nH
# Adjust Lj to tune f_01.
eig_qb.sim.setup.vars = Dict(
    Lj = '13 nH',   # single JJ inductance — adjust to tune f_01
    Cj = '2 fF',    # shunt capacitance
)

# ── HFSS box buffer ───────────────────────────────────────────────────────
eig_qb.sim.renderer.options['x_buffer_width_mm'] = 0.5
eig_qb.sim.renderer.options['y_buffer_width_mm'] = 0.5

print("EPRanalysis setup:")
eig_qb.sim.setup

EPRanalysis setup:


{'name': 'Qbit_Setup',
 'reuse_selected_design': True,
 'reuse_setup': True,
 'min_freq_ghz': 1.0,
 'n_modes': 1,
 'max_delta_f': 0.1,
 'max_passes': 15,
 'min_passes': 1,
 'min_converged': 2,
 'pct_refinement': 30,
 'basis_order': 1,
 'vars': {'Lj': '13 nH', 'Cj': '2 fF'}}

In [8]:
# ── Render design to HFSS ─────────────────────────────────────────────────
# Prerequisite: Ansys HFSS must be running with a valid licence.

# Get the renderer from the sim object
hfss = eig_qb.sim.renderer

#Connect to the running HFSS instance
hfss.start()

# Render only — no analysis launched
hfss.render_design(
    selection=['TC1'],
    open_pins=[],
    box_plus_buffer=True
)

eig_qb.sim.print_run_args()
print()
print("HFSS rendering complete.")
print()

# ── Confirm JJ object names that pyEPR will look for ─────────────────────
comp = 'TC1'
print(f"Expected HFSS JJ object names for component '{comp}':")
print(f"  rect : JJ_rect_Lj_{comp}_rect_jj1")
print(f"  line : JJ_Lj_{comp}_rect_jj1_")

INFO 07:52AM [connect_project]: Connecting to Ansys Desktop API...
INFO 07:52AM [load_ansys_project]: 	Opened Ansys App
INFO 07:52AM [load_ansys_project]: 	Opened Ansys Desktop v2024.2.0
INFO 07:52AM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/Kobi/Documents/Ansoft/
	Project:   Project34
INFO 07:52AM [connect_design]: No active design found (or error getting active design).
INFO 07:52AM [connect]: 	 Connected to project "Project34". No design detected


AttributeError: 'NoneType' object has no attribute 'draw_polyline'